In [1]:
from os import getenv
import time

from dotenv import load_dotenv
from openai import OpenAI

import pandas as pd

In [2]:
load_dotenv()

OPENAI_API_KEY = getenv("OPENAI_API_KEY")

In [3]:
# 데이터 로드
train_df = pd.read_excel("./dataset/train_dataset01.xlsx")
test_df = pd.read_excel("./dataset/test_dataset01.xlsx")
train_df.head()


,instruction,input,output
0,운동 초보라 자세 위주로 천천히 하고 싶어.,"{'성별': '남성', '몸무게': 107.7, 'squat_1RM': 320.0,...",NaN
1,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 86.17, 'squat_1RM': 190.0,...",NaN
2,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 64.9, 'squat_1RM': 160.0, ...",NaN
3,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 86.4, 'squat_1RM': 160.0, ...",NaN
4,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 92.55, 'squat_1RM': 220.0,...",NaN


In [4]:
try:
    client = OpenAI(
        api_key=OPENAI_API_KEY
    )
except Exception as e:
    print("모델 로드 오류 : ", e)

In [6]:
prompt =f"""
### Guidelines ###
[1] 당신은 긍정적이고, 유능한 헬스 트레이너입니다.
[2] 531 by jim wendler에 기반하여, 사용자에게 운동가이드라인을 제공하고 있습니다.
[3] 당신은 각 주차 별로 사용자들에게 중량을 추천해줘야 합니다. 추천하는 무게의 단위는 5kg입니다. 1의 자리에서 반올림해서 계산해주세요.
[4] 사용자의 1RM 입력에 대해 TM(실제 1RM의 90%)을 계산해서, TM을 기준으로 중량을 추천해주세요.
[5] 사용자가 특정 부위를 더욱 강화하길 윈하면, 조커 세트를 추가해줘야 합니다.
[6] 사용자가 특정 부위에 부상이 있다면, 중량을 반내림하여 낮춰야 합니다.
[7] 사용자의 운동 경력에 맞게 답변에 사용할 운동 용어를 조절하여, 사용자가 이해하기 쉬운 수준에서 답변해주세요.
[8] 예를 들어, 초급자에게는 '1RM' -> '들 수 있는 최대 무게'처럼 쉽게 풀어 설명해주세요.
[9] 사용자들에게 동기부여해주세요. 그러면 사람들은 당신을 더욱 신뢰하고, 연봉이 상승할지도 몰라요!

### Examples ###
<instruction>
요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.

<input>
'성별': '남성', '몸무게': 86.17, 'squat_1RM': 190.0, 'press_1RM': 80.5, 'bench_press_1RM': 115.0, 'deadlift_1RM': 212.5

<output>
허리가 안 좋다면 데드리프트를 조심스럽게 접근하는 것이 중요합니다. 대신 다른 운동을 통해 허리의 부담을 줄이면서도 효과적으로 운동할 수 있도록 도와드릴게요. 531 프로그램을 기반으로 주차별로 추천 중량을 안내해드리겠습니다. 이 프로그램은 4주 주기로 구성되어 있습니다.

### 1주차:
- **스쿼트**: 110회 kg, 5회 130, 5회 이상 145kg
- **밀리터리 프레스**: 5회 45kg, 5회 55kg, 5회 이상 60kg
- **벤치 프레스**: 5회 65g, 5회 80kg, 5회 이상 90kg
- **데드리프트**: 5회 115kg, 5회 135kg, 5회 이상 150kg (허리를 고려해 무게를 낮추었습니다)

### 2주차:
- **스쿼트**: 3회 120kg, 3회 130kg, 3회 이상 155kg
- **밀리터리 프레스**: 3회 50kg, 3회 60kg, 3회 이상 65kg
- **벤치 프레스**: 3회 70kg, 3회 80kg, 3회 이상 90kg
- **데드리프트**: 3회 125kg, 3회 140kg, 3회 이상 160kg (허리를 고려해 무게를 낮추었습니다)

### 3주차:
- **스쿼트**: 5회 130kg, 3회 145kg, 1회 이상 160kg
- **밀리터리 프레스**: 5회 55kg, 3회 60kg, 1회 이상 70kg
- **벤치 프레스**: 5회 80kg, 3회 90kg, 1회 이상 100kg
- **데드리프트**: 5회 130kg, 3회 150kg, 1회 이상 170kg (허리를 고려해 무게를 낮추었습니다)

### 4주차 (디로드 주):
- **스쿼트**: 5회 70kg, 5회 85kg, 5회 이상 100kg
- **밀리터리 프레스**: 5회 30kg, 5회 35kg, 5회 이상 45kg
- **벤치 프레스**: 5회 40g, 5회 50kg, 5회 이상 60kg
- **데드리프트**: 5회 65kg, 5회 85kg, 5회 이상 105kg (허리를 고려해 무게를 낮추었습니다)

- 모든 운동은 가벼운 중량으로 5회 3세트씩 진행합니다. 이때는 허리의 회복을 위해 데드리프트는 생략하거나 매우 가벼운 무게로 진행하세요.

운동 중 허리에 통증이 느껴진다면 즉시 중단하고 전문가의 상담을 받는 것이 중요합니다. 꾸준히 운동을 하다 보면 더 강해질 수 있습니다. 힘내세요! 당신의 건강 목표를 응원합니다! 💪
"""

In [ ]:
train_outputs =[]

for index, row in train_df.iterrows():
    instruction = row["instruction"]
    input = row["input"]

    user_input =f"""{instruction}
            
    [input]
    {input}
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": user_input}
            ],
            temperature=0.1
        )
        result = response.choices[0].message.content
    except Exception as e:
        print(f"index : {index} | ❌ ERROR : {e}")
        train_outputs.append(e)
    train_outputs.append(result)

    print(f"index : {index} | ✅ Good")

index : 0 | ✅ Good
index : 1 | ✅ Good
index : 2 | ✅ Good
index : 3 | ✅ Good
index : 4 | ✅ Good
index : 5 | ✅ Good
index : 6 | ✅ Good
index : 7 | ✅ Good
index : 8 | ✅ Good
index : 9 | ✅ Good
index : 10 | ✅ Good
index : 11 | ✅ Good
index : 12 | ✅ Good
index : 13 | ✅ Good


KeyboardInterrupt: 

In [ ]:
train_df["output"] = train_outputs

In [ ]:
train_df.to_excel("./dataset/train_dataset02.xlsx", index=False)

In [7]:
test_outputs =[]

for index, row in test_df.iterrows():
    instruction = row["instruction"]
    input = row["input"]

    user_input =f"""{instruction}
            
    [input]
    {input}
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": user_input}
            ],
            temperature=0.1
        )
        result = response.choices[0].message.content
    except Exception as e:
        print(f"index : {index} | ❌ ERROR : {e}")
        test_outputs.append(e)
    test_outputs.append(result)

    print(f"index : {index} | ✅ Good")

index : 0 | ✅ Good
index : 1 | ✅ Good
index : 2 | ✅ Good
index : 3 | ✅ Good
index : 4 | ✅ Good
index : 5 | ✅ Good
index : 6 | ✅ Good
index : 7 | ✅ Good
index : 8 | ✅ Good
index : 9 | ✅ Good
index : 10 | ✅ Good
index : 11 | ✅ Good
index : 12 | ✅ Good
index : 13 | ✅ Good
index : 14 | ✅ Good
index : 15 | ✅ Good
index : 16 | ✅ Good
index : 17 | ✅ Good
index : 18 | ✅ Good
index : 19 | ✅ Good
index : 20 | ✅ Good
index : 21 | ✅ Good
index : 22 | ✅ Good
index : 23 | ✅ Good
index : 24 | ✅ Good
index : 25 | ✅ Good
index : 26 | ✅ Good
index : 27 | ✅ Good
index : 28 | ✅ Good
index : 29 | ✅ Good
index : 30 | ✅ Good
index : 31 | ✅ Good
index : 32 | ✅ Good
index : 33 | ✅ Good
index : 34 | ✅ Good
index : 35 | ✅ Good
index : 36 | ✅ Good
index : 37 | ✅ Good
index : 38 | ✅ Good
index : 39 | ✅ Good
index : 40 | ✅ Good
index : 41 | ✅ Good
index : 42 | ✅ Good
index : 43 | ✅ Good
index : 44 | ✅ Good
index : 45 | ✅ Good
index : 46 | ✅ Good
index : 47 | ✅ Good
index : 48 | ✅ Good
index : 49 | ✅ Good
index : 50

In [8]:
test_df["output"] = test_outputs

In [9]:
test_df.to_excel("./dataset/test_dataset02.xlsx", index=False)